# NEXUS `llm` Extension -- End-to-End Demo

Runs **entirely from `nexus_continuous/llm/`** -- no other package files are
imported for logic (only `jax`/`numpy`/`matplotlib`, and `policies.common`
if present, with an in-package fallback if not). Nothing outside
`nexus_continuous/llm/` is created or modified by this notebook.

Steps:
1. Bootstrap `jax` (real if installed, else the private shim in `_jax_stub/`).
2. Generate a skillset with the **mock** LLM backend (seeded, deterministic --
   swap to `--backend hf`/`openai` for a real model with no other code changes).
3. Compile it with `interpreter.py` into a runnable policy module and sanity-check it.
4. Run a **multi-seed hand-written vs. LLM comparison** (via `mock_training.py`) and plot it.
5. Run the **interactive refinement loop** (propose -> train -> feedback -> revise)
   across iterations and seeds, and plot the results.


In [ ]:
import sys, os

NOTEBOOK_DIR = os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from nexus_continuous.llm.jax_bootstrap import ensure_jax
used_stub = ensure_jax()
print("Using bundled jax stub:" , used_stub)


## 1. Generate a skillset with the mock LLM backend

In [ ]:
from nexus_continuous.llm.client import LLMClient, LLMConfig, MockSkillGenerator
from nexus_continuous.llm.pipeline import generate_skillset

env_name = "CartpoleBalance"
fields = ("cart_position", "pole_angle", "cart_velocity", "pole_angular_velocity")
task_description = "Keep the pole upright and centered while minimizing oscillations."

seed = 0
client = LLMClient(LLMConfig(backend="mock", seed=seed), mock_generator=MockSkillGenerator(fields, seed=seed))
skillset = generate_skillset(
    env_name=env_name,
    observation_schema="\n".join(fields),
    task_description=task_description,
    client=client,
    allowed_fields=set(fields),
)
for s in skillset.skills:
    print(f"- {s.name}: activation_rule={s.activation_rule!r}, {len(s.reward_terms)} reward term(s)")


## 2. Compile with `interpreter.py` and sanity-check

In [ ]:
from dataclasses import asdict
import jax.numpy as jnp
from nexus_continuous.llm.interpreter import make_policy_module

policy_module = make_policy_module(asdict(skillset), field_names=fields)
print("Skills:", policy_module.SKILL_NAMES)

rng = np.random.default_rng(0)
obs = jnp.asarray(rng.standard_normal((8, len(fields))).astype("float32"))
action = jnp.asarray(rng.standard_normal((8, 1)).astype("float32"))
done = jnp.zeros((8,), dtype=bool)

rewards = policy_module.skill_rewards(obs, obs, action, None, done, None)
mask = policy_module.skill_mask(obs, None)
meta_policy = policy_module.symbolic_meta_policy(obs, None)

print("skill_rewards shape:", rewards.shape)
print("skill_mask shape:", mask.shape)
print("symbolic_meta_policy:", np.asarray(meta_policy))
print(policy_module.explain_policy())


## 3. Multi-seed hand-written vs. LLM comparison

Uses `nexus_continuous.llm.mock_training` (a deterministic, seeded stand-in for
real JAX training -- see its module docstring) so this runs anywhere.


In [ ]:
from nexus_continuous.llm.mock_training import mock_train_fn, hand_written_baseline_metrics

n_seeds = 5
hand_runs = [hand_written_baseline_metrics(env_name, seed=s) for s in range(n_seeds)]
llm_runs = [mock_train_fn(asdict(skillset), seed=s) for s in range(n_seeds)]

def summarize(runs):
    vals = [r["returns/env_reward_mean"] for r in runs]
    return {"mean": float(np.mean(vals)), "std": float(np.std(vals))}

hand_summary = summarize(hand_runs)
llm_summary = summarize(llm_runs)
print("Handwritten:", hand_summary)
print("LLM:        ", llm_summary)


In [ ]:
from nexus_continuous.llm.plotting import plot_comparison

os.makedirs("plots", exist_ok=True)
path = plot_comparison(hand_summary, llm_summary, env_name, "plots/comparison.png")
plt.figure(figsize=(4.5, 4))
plt.imshow(plt.imread(path))
plt.axis("off")
plt.show()


## 4. Interactive refinement loop

Propose -> train (mock) -> feedback -> revise, across several iterations and seeds.


In [ ]:
from nexus_continuous.llm.pipeline import LLMSkillPipeline
from nexus_continuous.llm.refinement_loop import LLMRefinementLoop, RefinementConfig
from nexus_continuous.llm.mock_training import MockTrainer

def run_refinement(seed, iterations=6):
    client = LLMClient(LLMConfig(backend="mock", seed=seed), mock_generator=MockSkillGenerator(fields, seed=seed))
    pipeline = LLMSkillPipeline(client)
    loop = LLMRefinementLoop(pipeline, client)
    cfg = RefinementConfig(
        env_name=env_name,
        observation_schema="\n".join(fields),
        task_description=task_description,
        num_iterations=iterations,
        allowed_fields=set(fields),
    )
    return loop.run(cfg, MockTrainer(seed=seed))

results = {seed: run_refinement(seed).history for seed in range(4)}
for seed, history in results.items():
    vals = [rec.metrics["returns/env_reward_mean"] for rec in history]
    print(f"seed {seed}: env_reward_mean per iteration = {[round(v,2) for v in vals]}")


In [ ]:
from nexus_continuous.llm.plot import plot_refinement, plot_multi_seed_refinement

plot_refinement(results[0], "plots/refinement_seed0.png", title=f"{env_name} refinement (seed 0)")
path = plot_multi_seed_refinement(results, "plots/refinement_all_seeds.png", title=f"{env_name} refinement across seeds")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].imshow(plt.imread("plots/refinement_seed0.png")); axes[0].axis("off")
axes[1].imshow(plt.imread(path)); axes[1].axis("off")
plt.tight_layout()
plt.show()


## Takeaways

* Every step above runs using only code inside `nexus_continuous/llm/` --
  generation, validation, compilation, "training" (via the bundled mock
  trainer), comparison, and refinement -- fully offline, fully seeded /
  reproducible, in well under a second per cell.
* Swapping to a real LLM is a one-line change (`LLMConfig(backend="hf"/"openai")`).
* Swapping to real training means writing a `train_fn(skillset) -> metrics`
  that calls your real trainer instead of `MockTrainer` -- `LLMRefinementLoop.run`
  and the comparison cells above don't care which one they get.
* See `nexus_continuous/llm/README.md` for the full module overview and
  `prompts.md` for the exact LLM prompt templates used.
